### Imports

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.dataset import ImageDataset

In [ ]:
from src.dataset import ImageDataset
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode
from torch import Generator, nn, optim
from torch.utils.data import random_split, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
import torchvision
import torch
from sklearn.metrics import roc_auc_score
import time


### Setup Dataset & ResNet18 Model

In [ ]:
def return_resnet18_dataloader(root_train, root_test, batch_size, num_workers, seed):
    preprocess = models.ResNet18_Weights.IMAGENET1K_V1.transforms()

    dataset_train = ImageDataset(root=root_train, transform=preprocess, train=True)
    dataset_val = ImageDataset(root=root_train, transform=preprocess, train=True)
    dataset_test = ImageDataset(root=root_test, transform=preprocess, train=False)

    n = len(dataset_train)
    train_size = int(0.8 * n)
    val_size = n - train_size

    train_idx, val_idx = random_split(
        range(n),
        [train_size, val_size],
        generator=Generator().manual_seed(seed),
    )

    train_dataset = Subset(dataset_train, train_idx.indices)  
    val_dataset   = Subset(dataset_val, val_idx.indices)


    trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    testloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return trainloader, valloader, testloader

In [ ]:
root_train = os.path.join('..', 'data', 'ImageNetSubset')
root_test = os.path.join('..', 'data', 'collected_dataset')


trainloader, valloader, testloader = return_resnet18_dataloader(root_train=root_train, 
                                                                root_test=root_test, 
                                                                batch_size=4, 
                                                                num_workers=4, 
                                                                seed=42)

weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights, progress = True)

# changing the output layer to have 10 output nodes
num_in_features = model.fc.in_features
model.fc = nn.Linear(num_in_features, 10)

# freeze backbone
for param in model.parameters():
    param.requires_grad = False

# unfreeze last layer
for param in model.fc.parameters():
    param.requires_grad = True

## Finetuning the last layer

In [ ]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
def train_one_epoch(optimizer, criterion, model, device):
    running_loss = 0.
    last_loss = 0.

    for i, data in enumerate(trainloader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
            
        running_loss += loss.item()
        if i % 1000 == 999:
            last_loss = running_loss / 1000 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))

            running_loss = 0.

    return running_loss / len(trainloader)

In [ ]:
writer = SummaryWriter(log_dir=f'./runs/ResNet18/exp_{int(time.time())}')
early_stopper = EarlyStopper(patience=5, min_delta=0.1)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
EPOCHS = 300
best_vloss = 1_000_000.0

model.to(device)
print("Device:", device)

#for epoch in range(EPOCHS):
for epoch in range(EPOCHS):
    print(f'\n{epoch+1}. Epoch:')
    # --- Training ---
    model.train()
    avg_loss = train_one_epoch(optimizer, criterion, model, device)

    # --- Validation ---
    model.eval()
    running_vloss = 0.0
    
    y = []
    y_hat = []
    y_hat_prob = []
    
    with torch.no_grad():
        for i, vdata in enumerate(valloader):
            vinputs, vlabels = vdata
            vinputs, vlabels = vinputs.to(device), vlabels.to(device)
            voutputs = model(vinputs)
            vloss = criterion(voutputs, vlabels)
            running_vloss += vloss

            # for other metrics
            probs = torch.softmax(voutputs, dim=1)
            pred = probs.argmax(dim=1)

            y.append(vlabels.cpu())
            y_hat.append(pred.cpu())
            y_hat_prob.append(probs.cpu())

    y = torch.cat(y)
    y_hat = torch.cat(y_hat)
    y_hat_prob = torch.cat(y_hat_prob)
    
    val_roc_auc = roc_auc_score(y.numpy(), y_hat_prob.numpy(), multi_class='ovr', average='macro')    
    val_acc = (y == y_hat).float().mean().item()
    
    avg_vloss = running_vloss / (i + 1)
    
    print(f'  val_acc: {val_acc}  val_roc_auc: {val_roc_auc}')
    print(f'  avg_loss_train: {avg_loss}  avg_loss_val: {avg_vloss}')

    # --- TensorBoard Logging ---
    writer.add_scalar('Loss/train_epoch', avg_loss, epoch)
    writer.add_scalar('Loss/val_epoch', avg_vloss, epoch)
    writer.add_scalar('Accuracy/val_epoch', val_acc, epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'], epoch)
    writer.add_scalar('AUC_ROC/val_epoch', val_roc_auc, epoch)

    scheduler.step(avg_vloss)

    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        PATH = 'ResNet18.pth'
        torch.save(model.state_dict(), PATH)

    if early_stopper.early_stop(avg_vloss):
        print('Stopped\nMin Validation Loss:', early_stopper.min_validation_loss)
        break

    if val_acc >= 0.9999:
        print('Stopped: Validation Accuratcy at 0.9999')


print('Finished Training')
writer.close()